# 2. MCP原理与工具适配

第一节连接了官方文档 MCP，并发现返回的工具也是 `StructuredTool`。这与第五章用 `@tool` 创建的本地工具是什么关系？`MCPAdapter` 到底适配了什么？

本节让**同一个加法函数**走两条路径：本地 LangChain 工具，以及通过 MCP 发现、适配和调用的工具。

学习目标：

1. 看清 Host、Client、Server 的职责。
2. 区分 MCP 工具描述、LangChain 可执行工具和工具执行结果。
3. 观察字段适配、工具定义进入模型，以及请求和结果的完整流转。

前面的实验使用进程内 MCP 连接，不需要模型密钥或外部网络；2.8 再复用第一节配置，调用真实聊天模型。服务脚本的编写和启动方式在第三、第四节继续展开。

## 2.1 Host、Client、Server分别做什么

![MCP 的 Host、Client、Server](assets/03-mcp-host-client-server.svg)

| 角色 | 本例中的对象或代码 | 职责 |
|---|---|---|
| Host | notebook 中的应用代码与 LangChain Agent | 组织模型、工具和对话，决定如何使用客户端 |
| Client | `fastmcp.Client`，或 MCPAdapter 内部的客户端 | 发现工具，向 MCP 服务发起调用 |
| Server | `FastMCP("Calculator")` | 提供工具描述，执行注册的业务函数 |

本节三个角色都在一个 Python 进程里，仍通过 MCP 客户端与服务交互。角色表示职责，不能直接等同于三台机器。进程内连接也不会启动额外子进程或 HTTP 端口。

## 2.2 环境准备

使用项目 `.venv` 与已有的 `langchain[mcp]` 依赖。Jupyter 中使用顶层 `await`；API 基础可回看 [第一节](01_agent连接MCP.ipynb) 和 [第五章 03 的 3.1.3](../Chapter05_tools/03_tools_decorator.ipynb)。

先导入组件，打印当前版本。`langchain.mcp` 的 beta 提示与第一节相同。

In [2]:
import json
import os
import sys
from importlib.metadata import version
from pathlib import Path

from dotenv import load_dotenv
from fastmcp import Client, FastMCP
from langchain.agents import create_agent
from langchain.mcp import MCPAdapter, as_langchain_tool
from langchain.tools import tool
from langchain_core.messages import AIMessage, ToolMessage
from langchain_core.tools import BaseTool, StructuredTool
from langchain_core.utils.function_calling import convert_to_openai_tool
from langchain_deepseek import ChatDeepSeek

PROJECT_ROOT = next(
    (path for path in [Path.cwd(), *Path.cwd().parents]
     if (path / "pyproject.toml").is_file()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("请从 lang-chain-learn 项目或其子目录运行本笔记。")

ENV_FILE = PROJECT_ROOT / "conf" / ".env"
load_dotenv(dotenv_path=ENV_FILE, override=True)

print("Python:", sys.executable)
for package in ["langchain", "fastmcp", "mcp"]:
    print(f"{package}: {version(package)}")

/var/folders/14/yh05p4nj2z70mjs568rzs9gh0000gn/T/ipykernel_83679/1386886080.py:10: LangChainBetaWarning: `langchain.mcp` is in beta. It is actively being worked on, so the API may change.
  from langchain.mcp import MCPAdapter, as_langchain_tool


Python: /Users/skk/Developer/lang-chain-learn/.venv/bin/python
langchain: 1.4.3
fastmcp: 4.1.0
mcp: 2.3.0


## 2.3 同一个函数，形成两条工具路径

![本地工具与 MCP 工具的两条路径](assets/04-local-and-mcp-tool-paths.svg)

业务函数只写一次。`tool(add)` 是 `@tool` 的等价写法，它创建本地工具；`calc_server.tool(add)` 将同一函数注册到 MCP 服务。

调用日志用于观察函数什么时候真正执行。注册工具、生成 Schema 都不应该产生加法调用。

In [ ]:
business_calls = []


def add(a: int, b: int) -> int:
    """计算两个整数之和。"""
    result = a + b
    business_calls.append({"a": a, "b": b, "result": result})
    return result


local_add = tool(add)
# 将add注册到FastMcp
# calc_server是Sever层
calc_server = FastMCP("Calculator")
calc_server.tool(add)

print("本地工具类型:", type(local_add).__name__)
print("本地工具 func 是原函数:", local_add.func is add)
print("MCP 服务名称:", calc_server.name)
print("定义和注册后，业务函数执行次数:", len(business_calls))
print(type(await calc_server.get_tool("add")))

本地工具类型: StructuredTool
本地工具 func 是原函数: True
MCP 服务名称: Calculator
定义和注册后，业务函数执行次数: 0
<class 'fastmcp.tools.function_tool.FunctionTool'>


## 2.4 从MCP工具描述到LangChain工具

### 2.4.1 先看客户端发现的原始工具信息

`Client.list_tools()` 返回 MCP 协议的工具描述列表。第一节的 `MCPAdapter.list_tools()` 在这些描述之上，又做了一层适配。

下面创建客户端并发现 `add`。返回对象包含名称、描述、输入 Schema 等数据。

In [4]:
client = Client(calc_server)

async with client:
    protocol_tools = await client.list_tools()

print(f"protocol_tools的类型是:{type(protocol_tools[0])}")
protocol_add = next(item for item in protocol_tools if item.name == "add")

print("原始描述类型:", f"{type(protocol_add).__module__}.{type(protocol_add).__name__}")
print("name:", protocol_add.name)
print("description:", protocol_add.description)
print("input_schema:")
print(json.dumps(protocol_add.input_schema, ensure_ascii=False, indent=2))
print("output_schema:")
print(json.dumps(protocol_add.output_schema, ensure_ascii=False, indent=2))
print("是 LangChain BaseTool:", isinstance(protocol_add, BaseTool))
print("描述对象有 func 属性:", hasattr(protocol_add, "func"))
print("发现工具后，业务函数执行次数:", len(business_calls))

protocol_tools的类型是:<class 'mcp_types._types.Tool'>
原始描述类型: mcp_types._types.Tool
name: add
description: 计算两个整数之和。
input_schema:
{
  "type": "object",
  "additionalProperties": false,
  "properties": {
    "a": {
      "type": "integer"
    },
    "b": {
      "type": "integer"
    }
  },
  "required": [
    "a",
    "b"
  ]
}
output_schema:
{
  "properties": {
    "result": {
      "type": "integer"
    }
  },
  "required": [
    "result"
  ],
  "type": "object",
  "x-fastmcp-wrap-result": true
}
是 LangChain BaseTool: False
描述对象有 func 属性: False
发现工具后，业务函数执行次数: 0


这里的 `mcp.types.Tool` 是协议描述对象。它保存参数说明，并没有把 Python 的 `add` 函数交给客户端。

输入 Schema 说明 `a`、`b` 是整数；本例 FastMCP 根据返回类型 `int` 生成的输出 Schema 用 `result` 字段表示整数结果。

### 2.4.2 手动打开适配过程

![MCP 工具字段如何适配到 StructuredTool](assets/05-mcp-adapter-field-mapping.svg)

使用公开函数 `as_langchain_tool` 转换一个工具，观察它的两个输入与返回值：

| 项目 | 内容 |
|---|---|
| `tool` 参数 | 客户端发现的 MCP 工具描述，本例是 `protocol_add` |
| `client` 参数 | 调用该工具的 FastMCP Client |
| 等待后的返回值 | 符合 BaseTool 接口的工具；当前实际类型是 StructuredTool |

进入客户端上下文后完成适配，让适配器能够读取服务身份。此时仍不执行加法。

```python
as_langchain_tool(
    tool: Tool, #client.list_tools的返回值
    client: Client[Any] | ClientGroup,
) -> BaseTool
```

In [5]:
async with client:
    manually_adapted_add = await as_langchain_tool(protocol_add, client)

print("适配后类型:", type(manually_adapted_add).__name__)
print("符合 BaseTool 接口:", isinstance(manually_adapted_add, BaseTool))
print("name:", manually_adapted_add.name)
print("description:", manually_adapted_add.description)
print("args_schema 是字典:", isinstance(manually_adapted_add.args_schema, dict))
print("func:", manually_adapted_add.func)
print("coroutine 名称:", manually_adapted_add.coroutine.__name__)
print("response_format:", manually_adapted_add.response_format)
print("metadata:")
print(json.dumps(manually_adapted_add.metadata, ensure_ascii=False, indent=2))
print("适配后，业务函数执行次数:", len(business_calls))

适配后类型: StructuredTool
符合 BaseTool 接口: True
name: add
description: 计算两个整数之和。
args_schema 是字典: True
func: None
coroutine 名称: call_tool
response_format: content_and_artifact
metadata:
{
  "mcp": {
    "tool": {
      "_meta": {
        "fastmcp": {
          "tags": []
        }
      }
    },
    "server": {
      "name": "Calculator",
      "version": "4.1.0"
    }
  }
}
适配后，业务函数执行次数: 0


`func=None`，`coroutine` 保存的是请求客户端的包装函数。调用它时，才通过 Client 请求服务端执行 `add`。

**也就是说,client将 Tool包装为异步函数**

### 2.4.3 MCPAdapter如何批量完成这些步骤

第一节的 `MCPAdapter.list_tools()` 大致完成：客户端发现描述 → 逐个适配 → 返回 LangChain 工具列表。连接管理等细节由适配器与 FastMCP 处理。

对同一个服务使用 MCPAdapter，再与手动适配后的对象比较。

In [6]:
async with MCPAdapter(calc_server) as adapter:
    langchain_tools = await adapter.list_tools()

mcp_add = next(item for item in langchain_tools if item.name == "add")

print("返回容器:", type(langchain_tools).__name__)
print("实际元素类型:", type(mcp_add).__name__)
print("两种适配入口的名称一致:", mcp_add.name == manually_adapted_add.name)
print("两种适配入口的 Schema 一致:", mcp_add.args_schema == manually_adapted_add.args_schema)
print("本地工具与 MCP 工具都属于 StructuredTool:",
      isinstance(local_add, StructuredTool) and isinstance(mcp_add, StructuredTool))
print("本地工具执行函数:", local_add.func.__name__)
print("MCP 工具执行包装函数:", mcp_add.coroutine.__name__)
print("再次发现和适配后，业务函数执行次数:", len(business_calls))
print(mcp_add.name)

返回容器: list
实际元素类型: StructuredTool
两种适配入口的名称一致: True
两种适配入口的 Schema 一致: True
本地工具与 MCP 工具都属于 StructuredTool: True
本地工具执行函数: add
MCP 工具执行包装函数: call_tool
再次发现和适配后，业务函数执行次数: 0
add


## 2.5 工具信息如何到达模型

模型接收的是工具定义：名称、描述、参数格式。它根据这些信息填写 ToolCall，业务函数的执行代码由应用和服务负责。

先用第五章学过的 `convert_to_openai_tool` 查看兼容工具调用的定义格式。这一步只做格式转换，不请求模型或执行工具。

In [7]:
local_definition = convert_to_openai_tool(local_add)
mcp_definition = convert_to_openai_tool(mcp_add)

for label, definition in [("本地工具定义", local_definition), ("MCP 工具定义", mcp_definition)]:
    print(label)
    print(json.dumps(definition, ensure_ascii=False, indent=2))
    print()

print("转换工具定义后，业务函数执行次数:", len(business_calls))

本地工具定义
{
  "type": "function",
  "function": {
    "name": "add",
    "description": "计算两个整数之和。",
    "parameters": {
      "properties": {
        "a": {
          "type": "integer"
        },
        "b": {
          "type": "integer"
        }
      },
      "required": [
        "a",
        "b"
      ],
      "type": "object"
    }
  }
}

MCP 工具定义
{
  "type": "function",
  "function": {
    "name": "add",
    "description": "计算两个整数之和。",
    "parameters": {
      "type": "object",
      "additionalProperties": false,
      "properties": {
        "a": {
          "type": "integer"
        },
        "b": {
          "type": "integer"
        }
      },
      "required": [
        "a",
        "b"
      ]
    }
  }
}

转换工具定义后，业务函数执行次数: 0


两份定义中的主要参数一致，但不要据此认为所有字段都完全相同。本例 MCP Schema 还声明了 `additionalProperties=False`，表示不接受 Schema 之外的字段；本地自动推导的 Schema 没有显式声明这一项。

逐项比较模型会用到的信息：

In [8]:
local_function = local_definition["function"]
mcp_function = mcp_definition["function"]
local_parameters = local_function["parameters"]
mcp_parameters = mcp_function["parameters"]

print("名称相同:", local_function["name"] == mcp_function["name"])
print("描述相同:", local_function["description"] == mcp_function["description"])
print("参数名称与类型相同:", local_parameters["properties"] == mcp_parameters["properties"])
print("必填项相同:", local_parameters["required"] == mcp_parameters["required"])
print("本地 additionalProperties:", local_parameters.get("additionalProperties", "未显式声明"))
print("MCP additionalProperties:", mcp_parameters.get("additionalProperties", "未显式声明"))
print("工具定义中的字段:", list(mcp_function))

名称相同: True
描述相同: True
参数名称与类型相同: True
必填项相同: True
本地 additionalProperties: 未显式声明
MCP additionalProperties: False
工具定义中的字段: ['name', 'description', 'parameters']


## 2.6 调用请求与结果如何流转

![工具定义、调用请求和结果返回的过程](assets/06-mcp-discovery-and-execution.svg)

### 2.6.1 分别直接调用本地工具、MCP客户端和适配工具

先不经过模型，让三条调用都使用 `a=2`、`b=3`。

| 调用入口 | 输入 | 等待后得到什么 |
|---|---|---|
| `local_add.invoke(input)` | 业务参数字典 | 本例业务函数返回的整数 |
| `await client.call_tool(name, arguments)` | MCP 工具名与业务参数字典 | FastMCP 的 CallToolResult |
| `await mcp_add.ainvoke(input)` | 业务参数字典 | 适配后的内容块列表 |

`CallToolResult.data` 是 FastMCP 提供的 Python 值视图；`content` 是内容块，`structured_content` 是结构化内容。调用前后的计数可以证明：执行发生在调用阶段。

In [9]:
arguments = {"a": 2, "b": 3}

local_value = local_add.invoke(arguments)
async with client:
    protocol_result = await client.call_tool(name="add", arguments=arguments)
adapted_content = await mcp_add.ainvoke(arguments)

print("本地返回:", type(local_value).__name__, local_value)
print("MCP 客户端返回类型:", type(protocol_result).__name__)
print("MCP data:", protocol_result.data)
print("MCP content:", [(type(block).__name__, block.text) for block in protocol_result.content])
print("MCP structured_content:", protocol_result.structured_content)
print("适配工具返回类型:", type(adapted_content).__name__)
print("适配后文本:", [block["text"] for block in adapted_content if block["type"] == "text"])
print("业务函数执行次数:", len(business_calls))
print("调用日志:", business_calls)

本地返回: int 5
MCP 客户端返回类型: CallToolResult
MCP data: 5
MCP content: [('TextContent', '5')]
MCP structured_content: {'result': 5}
适配工具返回类型: list
适配后文本: ['5']
业务函数执行次数: 3
调用日志: [{'a': 2, 'b': 3, 'result': 5}, {'a': 2, 'b': 3, 'result': 5}, {'a': 2, 'b': 3, 'result': 5}]


三次调用的业务结果都是 5，返回对象的形状各有不同。适配器将 MCP 内容转换为 LangChain 内容块，并准备好附件的处理方式。

### 2.6.2 用完整ToolCall观察ToolMessage

Agent 使用完整 ToolCall 调用工具。`args` 是业务参数，`id` 用于对应请求与工具消息。沿用第一节的做法，手工填写两份调用，比较消息的文本、附件和调用 ID。

In [10]:
local_call = {
    "name": local_add.name,
    "args": {"a": 2, "b": 3},
    "id": "local_call_1",
    "type": "tool_call",
}
mcp_call = {
    "name": mcp_add.name,
    "args": {"a": 2, "b": 3},
    "id": "mcp_call_1",
    "type": "tool_call",
}

local_message = local_add.invoke(local_call)
mcp_message = await mcp_add.ainvoke(mcp_call)

for label, message in [("本地", local_message), ("MCP", mcp_message)]:
    print(label, type(message).__name__)
    print("content 类型:", type(message.content).__name__)
    print("text:", message.text)
    print("tool_call_id:", message.tool_call_id)
    print("artifact:", message.artifact)
    print()

print("两种消息的文本结果一致:", local_message.text == mcp_message.text)
print("MCP 附件中的业务结果:", mcp_message.artifact["structured_content"]["result"])
print("最新两次业务调用:", business_calls[-2:])

本地 ToolMessage
content 类型: str
text: 5
tool_call_id: local_call_1
artifact: None

MCP ToolMessage
content 类型: list
text: 5
tool_call_id: mcp_call_1
artifact: {'structured_content': {'result': 5}}

两种消息的文本结果一致: True
MCP 附件中的业务结果: 5
最新两次业务调用: [{'a': 2, 'b': 3, 'result': 5}, {'a': 2, 'b': 3, 'result': 5}]


本地工具的 `content` 是字符串，MCP 工具的 `content` 是内容块列表，但这里的 `message.text` 都能读出 `5`。本例 MCP 的结构化结果被保存在 `artifact["structured_content"]` 中。

业务函数仍只接收 `a`、`b`。ToolCall 的调用 ID 用于 LangChain 消息包装，没有成为业务函数的新参数。

## 2.7 修改一个条件：换输入，结果怎样变化

把参数改为 `a=10`、`b=-3`，工具定义和调用入口保持不变。用新的调用 ID 再观察一次结果。

In [11]:
changed_arguments = {"a": 10, "b": -3}
changed_local_value = local_add.invoke(changed_arguments)
changed_mcp_message = await mcp_add.ainvoke({
    "name": mcp_add.name,
    "args": changed_arguments,
    "id": "mcp_changed_input",
    "type": "tool_call",
})

print("新的输入:", changed_arguments)
print("本地业务结果:", changed_local_value)
print("MCP 业务结果:", changed_mcp_message.artifact["structured_content"]["result"])
print("MCP 调用 id:", changed_mcp_message.tool_call_id)
print("当前工具仍叫:", mcp_add.name)
print("最新两次业务调用:", business_calls[-2:])

新的输入: {'a': 10, 'b': -3}
本地业务结果: 7
MCP 业务结果: 7
MCP 调用 id: mcp_changed_input
当前工具仍叫: add
最新两次业务调用: [{'a': 10, 'b': -3, 'result': 7}, {'a': 10, 'b': -3, 'result': 7}]


从日志中可以看到新的实际调用。工具描述规定怎样填写参数，业务结果需要等函数执行后才能得到。

## 2.8 接入真实模型：Agent的工具调用循环是否相同

前面的实验已完成工具发现与适配。现在复用第一节的 `ChatDeepSeek`、`deepseek-v4-flash` 和 `conf/.env`，观察同一个模型分别使用本地工具与 MCP 工具。

本小节会请求模型；两次 Agent 调用通常各包含“提出工具调用”和“根据结果回答”两次模型请求。模型配置与工具定义如何绑定，先分开观察。

In [12]:
if not os.getenv("DEEPSEEK_API_KEY"):
    raise ValueError(f"请在 {ENV_FILE} 中配置 DEEPSEEK_API_KEY，再运行真实模型部分。")

model = ChatDeepSeek(
    model="deepseek-v4-flash",
    api_key=os.environ["DEEPSEEK_API_KEY"],
    base_url=os.getenv("DEEPSEEK_BASE_URL"),
    extra_body={"thinking": {"type": "disabled"}},
    timeout=60,
    max_retries=1,
)

bound_model = model.bind_tools([mcp_add])
print("绑定后的工具定义:")
print(json.dumps(bound_model.kwargs["tools"], ensure_ascii=False, indent=2))
print("绑定工具后，业务函数执行次数:", len(business_calls))

绑定后的工具定义:
[
  {
    "type": "function",
    "function": {
      "name": "add",
      "description": "计算两个整数之和。",
      "parameters": {
        "type": "object",
        "additionalProperties": false,
        "properties": {
          "a": {
            "type": "integer"
          },
          "b": {
            "type": "integer"
          }
        },
        "required": [
          "a",
          "b"
        ]
      }
    }
  }
]
绑定工具后，业务函数执行次数: 7


`bind_tools` 把工具定义存入模型调用配置，此时还没发送模型请求，也没执行加法。实际调用模型时，SDK 会把这份定义与消息一起交给模型服务。

两种工具都按相同的 `create_agent(model=..., tools=[...])` 方式接入。下面查看真实模型产生的 ToolCall 和对应的 ToolMessage。

In [13]:
USER_QUESTION = "请调用 add 工具计算 17 加 25，并简短回答结果。"
agent_results = {}

for label, selected_tool in [("本地工具", local_add), ("MCP 工具", mcp_add)]:
    agent = create_agent(
        model=model,
        tools=[selected_tool],
        system_prompt="你是加法实验助手。计算任务必须先调用提供的 add 工具，依据工具结果用中文简短回答。",
    )
    response = await agent.ainvoke(
        {"messages": [{"role": "user", "content": USER_QUESTION}]},
        config={"recursion_limit": 10},
    )
    agent_results[label] = response

    print(label)
    for message in response["messages"]:
        if isinstance(message, AIMessage) and message.tool_calls:
            print("AIMessage.tool_calls:", json.dumps(message.tool_calls, ensure_ascii=False))
        elif isinstance(message, ToolMessage):
            print("ToolMessage:", message.name, message.tool_call_id, message.status)
            print("工具文本:", message.text)
            #artifact 是工具执行的"完整产物"，专门用来存放那些"不给模型看"的部分。
            print("工具附件:", message.artifact)
        elif isinstance(message, AIMessage):
            print("最终回答:", message.text)
    print()

本地工具
AIMessage.tool_calls: [{"name": "add", "args": {"a": 17, "b": 25}, "id": "call_00_zOPvZk2oz35OhkwlFoHw3322", "type": "tool_call"}]
ToolMessage: add call_00_zOPvZk2oz35OhkwlFoHw3322 success
工具文本: 42
工具附件: None
最终回答: 17 加 25 等于 42。

MCP 工具
AIMessage.tool_calls: [{"name": "add", "args": {"a": 17, "b": 25}, "id": "call_00_4t9wOve8VTrxFaBF893P5292", "type": "tool_call"}]
ToolMessage: add call_00_4t9wOve8VTrxFaBF893P5292 success
工具文本: 42
工具附件: {'structured_content': {'result': 42}}
最终回答: 17 + 25 = 42



模型每次生成的调用 ID 和最终措辞可能不同。比较工具消息和业务结果，判断调用路径是否真正完成。

In [14]:
for label, response in agent_results.items():
    requested_ids = [
        call["id"]
        for message in response["messages"]
        if isinstance(message, AIMessage)
        for call in message.tool_calls
    ]
    tool_messages = [message for message in response["messages"] if isinstance(message, ToolMessage)]
    returned_ids = [message.tool_call_id for message in tool_messages]
    print(label)
    print("工具请求次数:", len(requested_ids))
    print("成功工具消息数:", sum(message.status == "success" for message in tool_messages))
    print("请求与返回 id 对应:", requested_ids == returned_ids)
    print("工具结果:", [message.text for message in tool_messages])
    print()

print("模型实验的最新业务调用:", business_calls[-2:])

本地工具
工具请求次数: 1
成功工具消息数: 1
请求与返回 id 对应: True
工具结果: ['42']

MCP 工具
工具请求次数: 1
成功工具消息数: 1
请求与返回 id 对应: True
工具结果: ['42']

模型实验的最新业务调用: [{'a': 17, 'b': 25, 'result': 42}, {'a': 17, 'b': 25, 'result': 42}]


## 2.9 小结与小练习

| 比较项 | 本地 LangChain 工具 | 适配后的 MCP 工具 |
|---|---|---|
| 工具类型 | 本例是 StructuredTool | 本例同样是 StructuredTool |
| 参数描述 | 根据函数签名生成 Pydantic 模型 | 来自服务端的 JSON Schema，经适配保存 |
| 执行实现 | `func` 直接调用本地函数 | `coroutine` 通过 Client 请求服务端 |
| 模型看到的定义 | 名称、描述、参数格式 | 同样提供这些信息，约束细节可能不同 |
| 完整 ToolCall 的返回 | ToolMessage | ToolMessage，MCP 结果可带结构化附件 |

本节应能解释这条链路：**发现工具描述 → 创建客户端包装工具 → 把定义交给模型 → 根据 ToolCall 执行 → 转换结果并包装消息**。

三个 `list_tools()` 入口名字相同，但所在层次和返回对象不同。本例的比较如下：

| 调用入口 | 所在层次 | 列表中 `add` 的实际类型 | 包含什么 | 能否直接传给 `model.bind_tools()` |
|---|---|---|---|---|
| `await calc_server.list_tools()` | FastMCP 服务端 | `fastmcp.tools.function_tool.FunctionTool` | 服务端工具定义与业务函数 `fn` | 不能，需要通过客户端发现并适配 |
| `await client.list_tools()` | MCP 客户端 | MCP 协议 `Tool`（本环境为 `mcp_types._types.Tool`） | 名称、描述、输入/输出 Schema 等协议数据 | 不能，需要用 `as_langchain_tool` 适配 |
| `await adapter.list_tools()` | LangChain 适配层 | `langchain_core.tools.StructuredTool` | 工具描述与通过 Client 调用服务的异步包装函数 | 可以，也可以传给 `create_agent()` |

`MCPAdapter.list_tools()` 完成了“客户端发现 → LangChain 工具适配”。`calc_server.list_tools()` 是查看服务端内部工具，不能代替它；直接把本例的 `FunctionTool` 交给 `bind_tools()` 会报 `ValueError: Unsupported function`。

修改 docstring 后，要重新创建本地工具、注册服务端工具、发现并适配 MCP 工具，再重新绑定模型，新的 `description` 才会沿这条链路传递。`bind_tools()` 只准备模型调用配置，不请求模型，也不执行业务函数。docstring 的变化不会改写函数体。

小练习：

1. 把 2.7 的输入改成另一组整数，观察定义、调用日志和结果分别发生什么变化。
2. 修改 `add` 的 docstring，从 2.3 开始重新执行，比较本地与 MCP 工具描述，以及模型绑定配置里的 `description`。
3. 在 2.8 比较两次运行的 ToolCall：哪些字段表示业务参数，哪个字段用来对应工具消息？

下一节进一步学习 FastMCP 服务开发，增加工具并整理为独立脚本；连接的进程与传输细节在第四节展开。

### 练习一

In [15]:
changed_arguments = {"a": 2, "b": -3}
changed_local_value = local_add.invoke(changed_arguments)
changed_mcp_message = await mcp_add.ainvoke({
    "name": mcp_add.name,
    "args": changed_arguments,
    "id": "mcp_changed_input",
    "type": "tool_call",
})

# 比较定义
local_definition = convert_to_openai_tool(local_add)
mcp_definition = convert_to_openai_tool(mcp_add)

for label, definition in [("本地工具定义", local_definition), ("MCP 工具定义", mcp_definition)]:
    print(label)
    print(json.dumps(definition, ensure_ascii=False, indent=2))
    print()


print("新的输入:", changed_arguments)
print("本地业务结果:", changed_local_value)
print("MCP 业务结果:", changed_mcp_message.artifact["structured_content"]["result"])
print("MCP 调用 id:", changed_mcp_message.tool_call_id)
print("当前工具仍叫:", mcp_add.name)
print("最新两次业务调用:", business_calls[-2:])

本地工具定义
{
  "type": "function",
  "function": {
    "name": "add",
    "description": "计算两个整数之和。",
    "parameters": {
      "properties": {
        "a": {
          "type": "integer"
        },
        "b": {
          "type": "integer"
        }
      },
      "required": [
        "a",
        "b"
      ],
      "type": "object"
    }
  }
}

MCP 工具定义
{
  "type": "function",
  "function": {
    "name": "add",
    "description": "计算两个整数之和。",
    "parameters": {
      "type": "object",
      "additionalProperties": false,
      "properties": {
        "a": {
          "type": "integer"
        },
        "b": {
          "type": "integer"
        }
      },
      "required": [
        "a",
        "b"
      ]
    }
  }
}

新的输入: {'a': 2, 'b': -3}
本地业务结果: -1
MCP 业务结果: -1
MCP 调用 id: mcp_changed_input
当前工具仍叫: add
最新两次业务调用: [{'a': 2, 'b': -3, 'result': -1}, {'a': 2, 'b': -3, 'result': -1}]


### 练习二：验证 docstring 的传递

这里沿用修改后的“计算两个整数之差。”，重新创建两种工具，并检查工具描述和模型绑定配置中的 `description` 是否一致。最后用 `a=2`、`b=-3` 调用两种工具：函数体仍是 `a + b`，结果应为 `-1`，说明修改描述没有改变业务逻辑。

本练习只绑定模型、直接调用工具，不向模型服务发送请求。


In [12]:
business_calls = []


def add(a: int, b: int) -> int:
    """计算两个整数之差。"""
    result = a + b
    business_calls.append({"a": a, "b": b, "result": result})
    return result


local_add = tool(add)
calc_server = FastMCP("Calculator")
calc_server.tool(add)

if not os.getenv("DEEPSEEK_API_KEY"):
    raise ValueError(f"请在 {ENV_FILE} 中配置 DEEPSEEK_API_KEY，再运行模型绑定部分。")

model1 = ChatDeepSeek(
    model="deepseek-v4-flash",
    api_key=os.environ["DEEPSEEK_API_KEY"],
    base_url=os.getenv("DEEPSEEK_BASE_URL"),
    extra_body={"thinking": {"type": "disabled"}},
    timeout=60,
    max_retries=1,
)

async with MCPAdapter(calc_server) as adapter:
    adapted_tools = await adapter.list_tools()
    mcp_add = next(t for t in adapted_tools if t.name == "add")

    print("本地工具类型:", type(local_add).__name__)
    print("本地工具 func 是原函数:", local_add.func is add)
    print("MCP 服务名称:", calc_server.name)
    print("MCP 工具类型:", type(mcp_add).__name__)

    # 同一个模型分别绑定两种工具，比较各自的配置。
    bound_model = model1.bind_tools([mcp_add])
    bound_model_local = model1.bind_tools([local_add])
    print("绑定后的 MCP 工具定义:")
    print(json.dumps(bound_model.kwargs["tools"], ensure_ascii=False, indent=2))
    print("绑定后的本地工具定义:")
    print(json.dumps(bound_model_local.kwargs["tools"], ensure_ascii=False, indent=2))

    descriptions = {
        "本地工具": local_add.description,
        "MCP 工具": mcp_add.description,
        "MCP 绑定配置": bound_model.kwargs["tools"][0]["function"]["description"],
        "本地绑定配置": bound_model_local.kwargs["tools"][0]["function"]["description"],
    }
    print("\n修改 docstring 后的描述校验:")
    for label, description in descriptions.items():
        assert description == "计算两个整数之差。"
        print(f"{label}: {description}（校验通过）")

    assert not business_calls
    print("发现、适配和绑定后的业务调用次数:", len(business_calls))

    docstring_arguments = {"a": 2, "b": -3}
    local_value = local_add.invoke(docstring_arguments)
    mcp_message = await mcp_add.ainvoke({
        "name": mcp_add.name,
        "args": docstring_arguments,
        "id": "mcp_docstring_check",
        "type": "tool_call",
    })
    mcp_value = mcp_message.artifact["structured_content"]["result"]
    assert local_value == mcp_value == -1
    print("调用输入:", docstring_arguments)
    print("本地业务结果:", local_value)
    print("MCP 业务结果:", mcp_value)
    print("业务调用日志:", business_calls)
    print("描述已改为“差”，函数体仍执行加法。")


本地工具类型: StructuredTool
本地工具 func 是原函数: True
MCP 服务名称: Calculator
MCP 工具类型: StructuredTool
绑定后的 MCP 工具定义:
[
  {
    "type": "function",
    "function": {
      "name": "add",
      "description": "计算两个整数之差。",
      "parameters": {
        "type": "object",
        "additionalProperties": false,
        "properties": {
          "a": {
            "type": "integer"
          },
          "b": {
            "type": "integer"
          }
        },
        "required": [
          "a",
          "b"
        ]
      }
    }
  }
]
绑定后的本地工具定义:
[
  {
    "type": "function",
    "function": {
      "name": "add",
      "description": "计算两个整数之差。",
      "parameters": {
        "properties": {
          "a": {
            "type": "integer"
          },
          "b": {
            "type": "integer"
          }
        },
        "required": [
          "a",
          "b"
        ],
        "type": "object"
      }
    }
  }
]

修改 docstring 后的描述校验:
本地工具: 计算两个整数之差。（校验通过）
MCP 工具: 计算两个整数之差。（校验通过）
MCP 

In [ ]:
# 总结
# 声明 Sever层
mcp_sever = FastMCP("Calcuate")
# 生成客户端层
Client(mcp_sever)
# 根据连接生成 LangChain下的Mcpadapter
MCP_URL = "https://docs.langchain.com/mcp"
docs_adapter = MCPAdapter(target=MCP_URL)

## 参考资料

- [第一节：Agent连接MCP](01_agent连接MCP.ipynb)
- [第五章：工具装饰器与 BaseTool / StructuredTool](../Chapter05_tools/03_tools_decorator.ipynb)
- [LangChain：MCP Tools](https://docs.langchain.com/oss/python/langchain/mcp/tools)
- [FastMCP：客户端调用工具](https://gofastmcp.com/clients/tools)